# Семинар 5: Рекуррентные нейронные сети для идентификации дикторов

## Введение

Рекуррентные нейронные сети (RNN) предназначены для обработки последовательных данных, таких как временные ряды, текст или аудио. В отличие от полносвязных и свёрточных сетей, RNN хранят внутреннее состояние (скрытое состояние), которое обновляется на каждом шаге последовательности.

В этом семинаре мы решим задачу **идентификации диктора** по аудиозаписи (классификация среди 28 дикторов). Сначала реализуем «ручную» **Vanilla RNN** на PyTorch, чтобы понять принцип работы, а затем воспользуемся готовой ячейкой **LSTM** и сравним результаты.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Попробуем использовать MPS (Apple Silicon) или CUDA, иначе CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Используемое устройство: {device}")

## 1. Знакомство с датасетом VCTK

**VCTK** (CSTR VCTK Corpus) – это набор записей голосов 110 дикторов, читающих одни и те же предложения на английском языке. Он широко используется для задач синтеза речи, идентификации дикторов и преобразования голоса.

Наша задача – по аудиофрагменту определить, какой диктор его произнёс (классификация на 28 классов).


In [ ]:
# %pip install gdown -q

In [ ]:
import gdown
import zipfile

FILE_ID = "1LHeFn_uLpgjsyVKvwhmtLxZ6lhaVnw7u"
output = "VCTK.zip"

if not os.path.exists(output):
    gdown.download(id=FILE_ID, output=output, quiet=False)

# Распаковка
with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall(".")

In [ ]:
# Пример структуры датасета после распаковки:
# VCTK/
#   p225_001.wav
#   p225_002.wav
#   ...

## 2. Параметры обработки аудио

Мы будем:
- Обрезать или дополнять паддингом до фиксированной длины (3 секунды → 48000 отсчётов).
- Извлекать **MFCC** (Mel-Frequency Cepstral Coefficients) – компактное представление спектра.
- Полученную последовательность (время × признаки) подавать на вход RNN/GRU/LSTM.


In [ ]:
SAMPLE_RATE = 16000
TARGET_LEN_SEC = 3
TARGET_LEN = int(SAMPLE_RATE * TARGET_LEN_SEC)   # 48000
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 512
WIN_LENGTH = 1024
N_MELS = 128

## 3. Класс датасета для VCTK

Класс должен:
- Сканировать папки дикторов в `VCTK/wav48/`.
- Каждому диктору присвоить уникальный целочисленный идентификатор (0 .. num_speakers-1).
- Для каждого аудиофайла:
  - загрузить его с помощью `torchaudio.load()`,
  - привести к моно и нужной частоте дискретизации,
  - обрезать/дополнить до `TARGET_LEN`,
  - нормализовать амплитуду,
- Вернуть последовательность признаков (seq_len, ) и метку диктора (`int`).


In [ ]:
from dataset import VCTKDataset

In [ ]:
# Укажите путь к распакованному датасету
DATASET_PATH = "./VCTK"
full_dataset = VCTKDataset(DATASET_PATH, TARGET_LEN)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size],
                                                 generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

NUM_CLASSES = len(full_dataset.id2label)

In [ ]:
mfcc = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "win_length": WIN_LENGTH,
        "n_mels": N_MELS,
        "center": False},
)

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs=3):
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        # --- TRAIN ---
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} | Train", leave=False)
        for waveform, labels in pbar:
            features = mfcc(waveform).to(device)
            B, F, T = features.shape
            features = features.reshape(B, T, F)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            t_loss += loss.item() * labels.size(0)
            t_correct += (logits.argmax(1) == labels).sum().item()
            t_total += labels.size(0)

        # --- VALIDATION ---
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} | Val  ", leave=False)
            for waveform, labels in pbar:
                features = mfcc(waveform).to(device)
                B, F, T = features.shape
                features = features.reshape(B, T, F)
                labels = labels.to(device)

                logits = model(features)
                loss = criterion(logits, labels)
                v_loss += loss.item() * labels.size(0)
                v_correct += (logits.argmax(1) == labels).sum().item()
                v_total += labels.size(0)

        history["train_loss"].append(t_loss/t_total)
        history["val_loss"].append(v_loss/v_total)
        history["train_acc"].append(t_correct/t_total)
        history["val_acc"].append(v_correct/v_total)
        print(f"Epoch {epoch+1:2d} | Train Acc: {t_correct/t_total:.2%} | Val Acc: {v_correct/v_total:.2%}")
    return history

## 4. Ручная реализация Vanilla RNN

Мы вручную создадим веса $W_{ih}, W_{hh}, b_{ih}, b_{hh}$ и реализуем цикл обновления скрытого состояния. Это поможет понять, как именно работает `nn.RNN` под капотом.

$$
h_t = \tanh\left(x_t W_{ih}^\top + b_{ih} + h_{t-1} W_{hh}^\top + b_{hh}\right)
$$

In [ ]:
class ManualRNNCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # Инициализация весов
        self.W_ih = None
        self.W_hh = None
        self.b_ih = None
        self.b_hh = None

    def forward(self, x, h_prev):
        # x: (batch, input_dim), h_prev: (batch, hidden_dim)
        pass

In [ ]:
class ManualVanillaRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.rnn_cell = ManualRNNCell(input_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        pass

In [ ]:
manual_rnn = ManualVanillaRNN(N_MFCC, hidden_dim=64, num_classes=NUM_CLASSES).to(device)
optimizer_rnn = optim.AdamW(manual_rnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
hist_manual = train_model(manual_rnn, train_loader, val_loader, optimizer_rnn, criterion, epochs=5)

## 5. LSTM (Long Short-Term Memory)

LSTM решает проблему затухания градиентов с помощью ячейки памяти $c_t$ и трёх ворот:
$$
\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) & \text{(Forget gate)} \\
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) & \text{(Input gate)} \\
\tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) & \text{(Candidate)} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t & \text{(Cell state update)} \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) & \text{(Output gate)} \\
h_t &= o_t \odot \tanh(c_t) & \text{(Hidden state)}
\end{aligned}
$$
PyTorch уже содержит оптимизированную реализацию `nn.LSTM`.

In [ ]:
class LSTMSpeakerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        # batch_first=True -> ожидаем (batch, seq, feature)
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        pass


In [ ]:
lstm_model = LSTMSpeakerModel(N_MFCC, hidden_dim=64, num_classes=NUM_CLASSES).to(device)
optimizer_lstm = optim.AdamW(lstm_model.parameters(), lr=1e-3)
hist_lstm = train_model(lstm_model, train_loader, val_loader, optimizer_lstm, criterion, epochs=5)

## 6. Bidirectional LSTM

Аудиосигнал содержит контекст как в прямом, так и в обратном направлении. Двусторонняя LSTM обрабатывает последовательность дважды:
$$
\vec{h}_t = \text{LSTM}_{\to}(x_t, \vec{h}_{t-1}), \quad \overleftarrow{h}_t = \text{LSTM}_{\leftarrow}(x_t, \overleftarrow{h}_{t+1})
$$
Итоговый вектор получается конкатенацией: $h_t = [\vec{h}_t ; \overleftarrow{h}_t]$. Размерность выхода удваивается.

In [ ]:
class BiLSTMSpeakerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        pass

In [ ]:
bilstm_model = BiLSTMSpeakerModel(N_MFCC, hidden_dim=64, num_classes=NUM_CLASSES).to(device)
optimizer_bilstm = optim.Adam(bilstm_model.parameters(), lr=1e-3)
hist_bilstm = train_model(bilstm_model, train_loader, val_loader, optimizer_bilstm, criterion, epochs=5)

## 7. Сравнение архитектур

Визуализируем динамику точности на валидации для всех трёх подходов. Обратите внимание, как быстро Vanilla RNN упирается в потолок из-за проблемы затухающих градиентов, тогда как LSTM и Bi-LSTM продолжают улучшать качество.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist_manual["val_acc"], marker="o", label="Vanilla RNN (Manual)")
plt.plot(hist_lstm["val_acc"], marker="s", label="LSTM (PyTorch)")
plt.plot(hist_bilstm["val_acc"], marker="^", label="Bi-LSTM")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.title("Speaker Identification: RNN vs LSTM vs Bi-LSTM")
plt.tight_layout()
plt.show()